# Stochastic Gradient Descent (SGD)

Gradient Descent is a powerful optimization framework, but its standard implementation—**Batch Gradient Descent (BGD)**—struggles to scale when dealing with massive datasets. **Stochastic Gradient Descent (SGD)** addresses these limitations by introducing randomness and updating parameters iteratively after inspecting individual data rows rather than the entire dataset.

---

## Summary Comparison of Gradient Descent Variants

| Variant | Data per Update | Updates per Epoch | Path to Convergence | RAM / Memory Requirements |
| :--- | :--- | :--- | :--- | :--- |
| **Batch Gradient Descent (BGD)** | Entire Dataset ($n$ rows) | $1$ | Smooth, stable, direct | **High** (must load all data into RAM for vectorization) |
| **Stochastic Gradient Descent (SGD)** | Single Row ($1$ sample) | $n$ | Noisy, zigzagging, stochastic | **Minimal** (only loads one row at a time) |
| **Mini-Batch Gradient Descent** | Batch Size $B$ (e.g., $32, 64$) | $n / B$ | Moderately noisy, balanced | **Moderate** (controllable batch size fits in RAM) |

---


## 1. The Pitfalls of Batch Gradient Descent

While Batch Gradient Descent is mathematically robust and converges in a smooth, direct path, it exhibits two severe bottlenecks as datasets grow:

### 1.1 High Computational Cost
To perform a single parameter update in BGD, the algorithm must calculate the derivative of the loss function for **every single row** in the dataset:

$$\nabla L(\beta) = -\frac{2}{n} \sum_{i=1}^{n} \mathbf{x}_i^T (y_i - \mathbf{x}_i \beta)$$

If a dataset contains $n = 100,000$ rows and $m = 100$ features, every single step requires $100,000 \times 100 = 10,000,000$ operations. Running BGD for $k = 1,000$ epochs results in $\mathcal{O}(k \cdot n \cdot m)$ total derivative calculations ($10^{10}$ operations), making it extremely slow.

### 1.2 Memory Limitations (Vectorization Bottleneck)
To make computations efficient, BGD relies on matrix multiplication (vectorization) which processes the entire input matrix $X$ at once:

$$\nabla L(\beta) = -\frac{2}{n} X^T (Y - X\beta)$$

This requires loading the entire matrix $X$ of size $n \times (m+1)$ and targets $Y$ into the system RAM. For massive datasets (e.g., millions of rows of clickstream data or images), this leads to **Out-Of-Memory (OOM)** errors, crashing the program.

## 2. What is Stochastic Gradient Descent?

> **Core Idea:** Instead of updating the parameter weights after evaluating the entire dataset, SGD updates the weights after seeing **each individual training row**.

### Key Advantages of SGD:

1. **Faster Convergence:** BGD updates the model parameters once per epoch. SGD updates them $n$ times per epoch. Even if the path is noisy, making millions of small updates allows the model to get close to the optimal solution in a fraction of the time.
2. **Memory Efficiency:** Since it only processes a single row (or small batch) at any given instance, SGD can train on datasets of arbitrary size (gigabytes to terabytes) that cannot fit in system RAM, making it highly scalable.

### The "Stochastic" (Random) Nature:
The word *stochastic* means random. In SGD, the dataset is shuffled, and the algorithm selects a training instance at random for each update step. 

Because each step is guided by only one row, the gradient is an approximation of the true overall gradient. Consequently, the loss does not decrease smoothly; instead, it bounces up and down, taking a **noisy, zigzagging path** toward the minimum. However, on average, the steps point toward the global optimum, and the algorithm converges to a near-optimal solution.

## 2.1 Mathematical Formulation

For a single observation $i$, let:
- $\mathbf{x}_i = [1, x_{i1}, x_{i2}, \dots, x_{im}]$ be the feature vector (with $1$ prepended for the intercept).
- $y_i$ be the actual target value.
- $\hat{y}_i = \mathbf{x}_i \beta$ be the predicted target value.

The loss function for this single sample is the squared error:

$$L_i(\beta) = (y_i - \hat{y}_i)^2 = (y_i - \mathbf{x}_i \beta)^2$$

Taking the derivative of $L_i$ with respect to the parameter vector $\beta$:

$$\nabla L_i(\beta) = -2 (y_i - \hat{y}_i) \mathbf{x}_i^T$$

### The SGD Update Rule:
For every row $i$ in a randomly shuffled dataset, we update the entire parameter vector $\beta$ immediately:

$$\beta_{new} = \beta_{old} - \eta \cdot \nabla L_i(\beta)$$

$$\beta_{new} = \beta_{old} + 2\eta (y_i - \hat{y}_i) \mathbf{x}_i^T$$

Where $\eta$ is the learning rate. In simple terms, for each feature coefficient $\beta_j$:

$$\beta_{j, new} = \beta_{j, old} + 2 \eta (y_i - \hat{y}_i) x_{ij}$$

---


## 3. Code Setup & Data Generation

To understand how SGD works, we will first generate a synthetic multi-feature dataset, split it into train/test sets, and establish a baseline using Scikit-Learn's standard linear regression.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Set display style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Generate synthetic regression dataset (150 samples, 3 features)
np.random.seed(42)
X, y = make_regression(n_samples=150, n_features=3, noise=10, random_state=42)

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training dataset size: {X_train.shape[0]} rows")
print(f"Testing dataset size:  {X_test.shape[0]} rows")


In [ ]:
# Fit OLS Linear Regression baseline to obtain standard parameters
ols = LinearRegression()
ols.fit(X_train, y_train)

print("OLS BASELINE PARAMETERS:")
print(f"Intercept:    {ols.intercept_:.6f}")
print(f"Coefficients: {ols.coef_}")
print(f"Test R2 Score: {r2_score(y_test, ols.predict(X_test)):.6f}")


## 4. Implementation from Scratch

Let's implement both **Batch Gradient Descent (BGD)** and **Stochastic Gradient Descent (SGD)** Regressors from scratch in Python to contrast their update mechanics. 

For SGD, we introduce **random shuffling** at the start of each epoch to ensure training rows are processed in a randomized order, satisfying the stochastic requirement.

In [ ]:
class BGDRegressor:
    """Vectorized Batch Gradient Descent Regressor"""
    def __init__(self, learning_rate=0.01, epochs=100):
        self.lr = learning_rate
        self.epochs = epochs
        self.coef_ = None
        self.intercept_ = None
        self.history = []  # Log parameters and overall MSE at each epoch
        
    def fit(self, X, y):
        X_new = np.insert(X, 0, 1, axis=1)  # Add intercept column of 1s
        n_samples, n_features = X_new.shape
        beta = np.zeros(n_features)  # Initialize parameters to 0
        
        for epoch in range(self.epochs):
            # Predict for the entire batch
            y_pred = np.dot(X_new, beta)
            
            # Calculate overall loss
            loss = np.mean((y - y_pred)**2)
            self.history.append((beta.copy(), loss))
            
            # Gradient calculation using the entire dataset
            d_beta = - (2 / n_samples) * np.dot(X_new.T, y - y_pred)
            
            # Simultaneous parameter update
            beta = beta - self.lr * d_beta
            
        self.intercept_ = beta[0]
        self.coef_ = beta[1:]
        
    def predict(self, X):
        return np.dot(X, self.coef_) + self.intercept_


class SGDRegressorCustom:
    """Stochastic Gradient Descent Regressor with Shuffling"""
    def __init__(self, learning_rate=0.01, epochs=100, learning_rate_decay=None):
        self.lr = learning_rate
        self.epochs = epochs
        self.decay = learning_rate_decay
        self.coef_ = None
        self.intercept_ = None
        self.history = []  # Log parameters and overall MSE at each epoch
        
    def fit(self, X, y):
        X_new = np.insert(X, 0, 1, axis=1)  # Add intercept column of 1s
        n_samples, n_features = X_new.shape
        beta = np.zeros(n_features)  # Initialize parameters to 0
        
        step_count = 0
        for epoch in range(self.epochs):
            # Shuffle the dataset indices at the start of each epoch
            indices = np.arange(n_samples)
            np.random.shuffle(indices)
            X_shuffled = X_new[indices]
            y_shuffled = y[indices]
            
            # Row-by-row updates
            for i in range(n_samples):
                step_count += 1
                xi = X_shuffled[i]
                yi = y_shuffled[i]
                
                # 1. Prediction for a single sample
                y_pred_i = np.dot(xi, beta)
                
                # 2. Gradient calculation for a single sample
                d_beta = -2 * (yi - y_pred_i) * xi
                
                # Apply optional learning rate schedule / decay
                if self.decay == 'inverse':
                    # Decay learning rate over update steps: lr_t = lr_init / (1 + 0.05 * step)
                    current_lr = self.lr / (1 + 0.05 * step_count)
                else:
                    current_lr = self.lr
                    
                # 3. Parameter update immediately
                beta = beta - current_lr * d_beta
                
            # Calculate overall loss at the end of the epoch
            y_pred_epoch = np.dot(X_new, beta)
            loss_epoch = np.mean((y - y_pred_epoch)**2)
            self.history.append((beta.copy(), loss_epoch))
            
        self.intercept_ = beta[0]
        self.coef_ = beta[1:]
        
    def predict(self, X):
        return np.dot(X, self.coef_) + self.intercept_


In [ ]:
# Fit custom BGD Regressor
bgd = BGDRegressor(learning_rate=0.1, epochs=100)
bgd.fit(X_train, y_train)
y_pred_bgd = bgd.predict(X_test)

# Fit custom SGD Regressor (Constant learning rate)
sgd = SGDRegressorCustom(learning_rate=0.01, epochs=100)
sgd.fit(X_train, y_train)
y_pred_sgd = sgd.predict(X_test)

# Print comparisons
print("="*60)
print("MODEL COEFFICIENTS COMPARISON")
print("="*60)
print(f"OLS Intercept:  {ols.intercept_:.6f} | Coefficients: {ols.coef_}")
print(f"BGD Intercept:  {bgd.intercept_:.6f} | Coefficients: {bgd.coef_}")
print(f"SGD Intercept:  {sgd.intercept_:.6f} | Coefficients: {sgd.coef_}")
print("-"*60)
print(f"OLS R2 Score: {r2_score(y_test, ols.predict(X_test)):.6f}")
print(f"BGD R2 Score: {r2_score(y_test, y_pred_bgd):.6f}")
print(f"SGD R2 Score: {r2_score(y_test, y_pred_sgd):.6f}")
print("="*60)


## 5. Visualizing Convergence: BGD vs. SGD

To inspect the differences in convergence behavior, we will generate a simple 1D dataset (so we only have slope $m$ and intercept $b$ as parameters) and plot their parameter trajectories over the contour plot of the MSE Loss Surface.

- **BGD Convergence:** Follows a smooth, direct trajectory perpendicular to the contours, moving directly toward the global minimum.
- **SGD Convergence:** Exhibits high volatility, zigzagging in a "noisy" path as it updates parameter values row-by-row.

In [ ]:
# Generate simple 1D dataset
np.random.seed(42)
X_simple = np.random.uniform(-2, 2, 80).reshape(-1, 1)
y_simple = 2.0 * X_simple.ravel() + 1.0 + np.random.normal(0, 0.4, 80)

# Train BGD and SGD models on this 1D dataset
bgd_simple = BGDRegressor(learning_rate=0.08, epochs=40)
bgd_simple.fit(X_simple, y_simple)

sgd_simple = SGDRegressorCustom(learning_rate=0.02, epochs=40)
sgd_simple.fit(X_simple, y_simple)

# Create grid of m (slope) and b (intercept) for loss surface contours
m_vals = np.linspace(-1, 5, 100)
b_vals = np.linspace(-1, 3, 100)
M, B = np.meshgrid(m_vals, b_vals)
Z = np.zeros(M.shape)

# Compute loss for each grid point
for i in range(len(m_vals)):
    for j in range(len(b_vals)):
        pred = M[j, i] * X_simple.ravel() + B[j, i]
        Z[j, i] = np.mean((y_simple - pred)**2)

# Plot contours and trajectories
fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))

# Plot 1: Contour Plot of Loss Surface
contours = axes[0].contour(M, B, Z, levels=30, cmap='viridis')
axes[0].clabel(contours, inline=True, fontsize=8)

# Extract parameter history (beta[1] = slope m, beta[0] = intercept b)
bgd_betas = np.array([h[0] for h in bgd_simple.history])
sgd_betas = np.array([h[0] for h in sgd_simple.history])

axes[0].plot(bgd_betas[:, 1], bgd_betas[:, 0], 'o-', color='#2ecc71', linewidth=2.5, markersize=5, label='BGD (Smooth Path)')
axes[0].plot(sgd_betas[:, 1], sgd_betas[:, 0], 'x-', color='#e74c3c', linewidth=1.5, markersize=5, alpha=0.8, label='SGD (Noisy Path)')
axes[0].scatter([2.0], [1.0], color='black', marker='*', s=200, zorder=5, label='Global Minimum')

axes[0].set_title('BGD vs SGD Parameter Path on Loss Surface Contour', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Slope (m)')
axes[0].set_ylabel('Intercept (b)')
axes[0].legend(loc='lower left')

# Plot 2: Epoch vs. Loss Curve
axes[1].plot([h[1] for h in bgd_simple.history], color='#2ecc71', linewidth=2.5, label='BGD Loss')
axes[1].plot([h[1] for h in sgd_simple.history], color='#e74c3c', linewidth=2.0, alpha=0.8, label='SGD Loss')
axes[1].set_title('BGD vs SGD MSE Loss per Epoch', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epochs')
axes[1].set_ylabel('Mean Squared Error')
axes[1].legend()

plt.tight_layout()
plt.show()


## 6. Advanced Concept: Learning Schedules

Because SGD updates weights sample-by-sample, it maintains a degree of volatility even when it gets near the optimal parameters. Instead of settling down directly at the minimum, it **fluctuates and oscillates** around it indefinitely if the learning rate $\eta$ is kept constant.

To solve this, we use a **Learning Schedule** (learning rate decay). The idea is to begin with a relatively large learning rate to speed up early exploration and escape local minima, and then systematically **decay/decrease the learning rate** over time so that the step sizes become smaller, allowing the parameters to stabilize and settle directly in the minimum.

### Common Learning rate decay formulas:

1. **Inverse Scaling (Standard in Scikit-Learn):**
   $$\eta_t = \frac{\eta_0}{t^p}$$
   Where $t$ is the step number, $\eta_0$ is the initial learning rate, and $p$ is the decay power (usually $0.25$ or $0.5$).
   
2. **Time-Based Decay:**
   $$\eta_t = \frac{\eta_0}{1 + \alpha \cdot t}$$
   Where $\alpha$ is the decay rate parameter.

Let's visualize the impact of using a learning schedule on SGD's convergence behavior.

In [ ]:
# Train SGD with constant learning rate
sgd_constant = SGDRegressorCustom(learning_rate=0.05, epochs=150)
sgd_constant.fit(X_train, y_train)

# Train SGD with learning rate decay (Inverse scaling schedule)
sgd_decayed = SGDRegressorCustom(learning_rate=0.05, epochs=150, learning_rate_decay='inverse')
sgd_decayed.fit(X_train, y_train)

# Plotting the comparison of MSE loss curves
plt.figure(figsize=(10, 5))
plt.plot([h[1] for h in sgd_constant.history], color='#e74c3c', alpha=0.7, label='SGD with Constant Learning Rate (0.05)')
plt.plot([h[1] for h in sgd_decayed.history], color='#3498db', alpha=0.9, label='SGD with Learning Rate Decay (Learning Schedule)')

plt.title('Impact of Learning Rate Schedule on SGD Stabilization', fontsize=12, fontweight='bold')
plt.xlabel('Epochs')
plt.ylabel('Overall Mean Squared Error')
plt.yscale('log')  # Log scale makes the oscillations and differences more readable
plt.legend()
plt.show()

print(f"Final MSE with Constant LR: {sgd_constant.history[-1][1]:.6f}")
print(f"Final MSE with Decayed LR:  {sgd_decayed.history[-1][1]:.6f}")


## 7. When to Use Stochastic Gradient Descent?

Stochastic Gradient Descent is preferred over Batch Gradient Descent in two primary scenarios:

### 7.1 Big Data Scalability
When your dataset has hundreds of thousands or millions of observations, loading it in memory for matrix math becomes impossible. SGD's single-row processing is a necessity for training on out-of-core, streamable datasets.

### 7.2 Escaping Local Minima (Non-Convex Surfaces)
In deep neural networks, the loss surface is highly non-convex, featuring thousands of local minima, hills, and flat saddle points. 
- **BGD** will calculate the exact gradient, moving smoothly and landing directly into whichever local minimum it is closest to, getting stuck permanently.
- **SGD's** updates are noisy and random. This noise introduces high volatility, causing the parameter updates to jump out of small, shallow local minima or flat saddle points. This increases the probability that the algorithm will continue searching and eventually find the global minimum, which is a major reason why SGD (and its variants like Adam/RMSprop) is the foundation of Deep Learning.

## 8. Scikit-Learn Implementation

Scikit-Learn provides a highly optimized version of Stochastic Gradient Descent for regression via the `SGDRegressor` class. It includes built-in support for learning schedules, stopping criteria, and regularization.

### Key Parameters of `SGDRegressor`:
- `loss`: The loss function to minimize. Defaults to `'squared_error'` (MSE).
- `penalty`: Regularization term (`'l2'`, `'l1'`, or `'elasticnet'`) to prevent overfitting.
- `alpha`: Regularization strength (higher values mean more regularization).
- `max_iter`: Maximum number of epochs (passes over the training data).
- `tol`: Stopping criterion tolerance. Training stops if loss doesn't decrease by more than `tol` for `n_iter_no_change` consecutive epochs.
- `learning_rate`: The learning rate schedule strategy:
  - `'constant'`: $\eta = \text{eta0}$
  - `'optimal'`: $\eta_t = \frac{1}{\alpha \cdot (t + t_0)}$
  - `'invscaling'`: $\eta_t = \frac{\text{eta0}}{t^p}$ (default schedule, with $p = 0.25$)
  - `'adaptive'`: Keeps learning rate constant at `eta0` as long as training loss decreases, then divides by 5 when progress stalls.
- `eta0`: The initial learning rate.

> **CRITICAL:** SGD is extremely sensitive to feature scaling. Always scale your features using standard standardization (mean 0, variance 1) before training an SGD model. Otherwise, parameters might diverge or converge extremely slowly.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDRegressor

# 1. Scaling the features (Required for SGD)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 2. Initialize and fit SGDRegressor with default invscaling learning schedule
sgd_reg = SGDRegressor(
    loss='squared_error', 
    max_iter=1000, 
    tol=1e-3, 
    learning_rate='invscaling', 
    eta0=0.01, 
    random_state=42
)
sgd_reg.fit(X_train_scaled, y_train)

# 3. Predict and evaluate
y_pred_scaled = sgd_reg.predict(X_test_scaled)
r2_sgd_scaled = r2_score(y_test, y_pred_scaled)

print("="*60)
print("SCIKIT-LEARN SGDREGRESSOR RESULTS (SCALED FEATURES)")
print("="*60)
print(f"Intercept:      {sgd_reg.intercept_[0]:.6f}")
print(f"Coefficients:   {sgd_reg.coef_}")
print(f"Test R2 Score:   {r2_sgd_scaled:.6f}")
print("="*60)


## Summary Checklist for Stochastic Gradient Descent

1. **Scale your Features:** SGD computes updates using feature values ($x_{ij}$). If features have unequal scales, the updates will be unbalanced, leading to instability. Use `StandardScaler` first.
2. **Shuffle your Data:** Always shuffle rows at each epoch. If data is ordered (e.g., sorted by age or target variable), SGD will update bias towards sequential clusters, preventing overall convergence.
3. **Use a Learning Schedule:** Constant learning rates cause SGD to oscillate around the minimum. Decay the learning rate over time to stabilize the parameters at the global minimum.
4. **Monitor Epoch-level Loss:** Watch the overall MSE loss over epochs to ensure the model converges and doesn't diverge due to an excessively high initial learning rate (`eta0`).